In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path


# =========================
# Set these folders
# =========================

input_folder = Path("Gen10")
output_folder = Path("Gen11")

output_folder.mkdir(parents=True, exist_ok=True)


# =========================
# Helper functions
# =========================

filename_pattern = re.compile(
    r"Gen(?P<gen>\d+)_Chain(?P<chain>\d+)",
    flags=re.IGNORECASE
)


def clean_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return x if x else np.nan


def extract_field(text, field):
    if pd.isna(text):
        return np.nan

    text = str(text)
    pattern = rf"{field}:\s*(.*?)(?:\s*\{{\"ImportId\"|$)"
    match = re.search(pattern, text, flags=re.S)

    if not match:
        return np.nan

    value = re.sub(r"\s+", " ", match.group(1)).strip()
    return value if value else np.nan


def parse_gen_and_chain(csv_path):
    """
    Extract Gen and Chain numbers from names like:
    Telephone+Game_Gen2_Chain10_May+8,+2026_11.25.csv
    """
    match = filename_pattern.search(csv_path.stem)

    if not match:
        return None, None

    current_gen = int(match.group("gen"))
    chain_number = int(match.group("chain"))

    return current_gen, chain_number


def create_next_generation(input_csv, output_folder):
    current_gen, chain_number = parse_gen_and_chain(input_csv)

    if current_gen is None or chain_number is None:
        print(f"Skipping, filename does not match Gen*_Chain* pattern: {input_csv.name}")
        return None

    next_gen = current_gen + 1

    raw = pd.read_csv(input_csv, dtype=str)

    question_row = raw.iloc[0]
    responses = raw.iloc[2:].copy().reset_index(drop=True)

    created_files = []

    for i, response_row in responses.iterrows():
        rows = []

        for col in raw.columns:
            task_text = question_row[col]
            answer = clean_value(response_row[col])

            if pd.isna(answer):
                continue

            if col.startswith("T0_Q"):
                original_word = extract_field(task_text, "Word")

                rows.append({
                    "Original word": original_word,
                    "Original description": np.nan,
                    "Guessed description": answer,
                    "Guessed word": np.nan,
                    "Type": 1
                })

            elif col.startswith("T1_Q"):
                original_description = extract_field(task_text, "Description")

                rows.append({
                    "Original word": np.nan,
                    "Original description": original_description,
                    "Guessed description": np.nan,
                    "Guessed word": answer.capitalize(),
                    "Type": 0
                })

        df_chain = pd.DataFrame(
            rows,
            columns=[
                "Original word",
                "Original description",
                "Guessed description",
                "Guessed word",
                "Type"
            ]
        )

        if len(responses) == 1:
            output_name = f"Gen_{next_gen}_Chain_{chain_number}.csv"
        else:
            output_name = f"Gen_{next_gen}_Chain_{chain_number}_Response_{i + 1}.csv"

            

        output_path = output_folder / output_name
        df_chain.to_csv(output_path, index=False)

        created_files.append(output_path)

    return created_files


# =========================
# Run for all CSV files
# =========================

all_created_files = []

csv_files = sorted(input_folder.glob("*.csv"))

for input_csv in csv_files:
    created = create_next_generation(input_csv, output_folder)

    if created is not None:
        all_created_files.extend(created)

print(f"Processed {len(csv_files)} CSV files.")
print(f"Created {len(all_created_files)} output files.")

for path in all_created_files:
    print(path)

Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_1.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_10.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_2.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_3.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_4.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_5.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_6.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_7.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_8.csv
Skipping, filename does not match Gen*_Chain* pattern: Gen_10_Chain_9.csv
Processed 20 CSV files.
Created 10 output files.
Gen11/Gen_11_Chain_10.csv
Gen11/Gen_11_Chain_1.csv
Gen11/Gen_11_Chain_2.csv
Gen11/Gen_11_Chain_3.csv
Gen11/Gen_11_Chain_4.csv
Gen11/Gen_11_Chain_5.csv
Gen11/Gen_11_Chain_6.csv
Gen11/Gen_11_Chain_7.csv
Gen11/Gen